# Stage 1 — Teacher batch labeling with Qwen2.5-VL-7B

This notebook runs the same labeling pipeline as [`scripts/run_teacher_batch_label.py`](../scripts/run_teacher_batch_label.py), cell by cell. It loads a slice of [Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023), prompts a Qwen2.5-VL-7B model with the product image and title, and writes a parquet of `{category, attributes, search_tags, description}` JSON per product.

The output parquet is the labeled corpus that Stage 2 distills into a 3B student. For the end-to-end story across all three stages, see the template's [`README.ipynb`](../README.ipynb).

In [ ]:
import os, sys, json
import ray
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "working_dir": ".",
        # vLLM 0.20+ moved TokensPrompt up from vllm.inputs.data to vllm.inputs.
        # Ray 2.55's batch LLM stage still imports via the old path, so every
        # worker patches it on startup. Must be set here on ray.init — adding it
        # to a per-actor runtime_env later is silently ignored by Ray's setup-hook
        # registry.
        "worker_process_setup_hook": "src._vllm_compat.patch",
    },
)
print("Cluster resources:", json.dumps(ray.cluster_resources(), indent=2))

## Load and preprocess the catalog

`ray.data.read_parquet` streams the Amazon-Reviews-2023 metadata directly from the Hugging Face Hub. The dataset arrives row-per-product with a struct of image URLs per row; the helpers below flatten that into the `{id, product_id, title, description, image_url, source}` schema the VLM stage expects.

A deterministic `id` (SHA-1 of title + image URL) lets Ray Data's `CheckpointConfig` resume mid-run if the cluster blips. UUIDs would regenerate on retry and the checkpoint would match zero rows.

### Sample row

Print the preprocessed schema, take one row, and display its image to confirm the load worked end-to-end before lighting up the GPUs.

In [ ]:
# load data

import os
import ray
from huggingface_hub import HfFileSystem
from typing import Optional
from src.load_catalog import load_amazon_reviews_2023
from PIL import Image
import hashlib
import io, requests

# data source params
category = "Electronics"
hf_path = f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw_meta_{category}"

# preprocess params
n_rows = 50
seed = 42

# paths
base_dir = "/mnt/cluster_storage/vlm-distillation-catalog-enrichment"
cache_path = f"{base_dir}/catalog_demo_{n_rows}.parquet"
checkpoint_path = f"{base_dir}/catalog_demo_{n_rows}_checkpoint"


def _extract_image_url(images_field) -> Optional[str]:
    """Pick the first 'large' image URL (or hi_res/thumb fallback) from Amazon's image struct.

    HF parquet stores `images` as a struct of parallel lists,
    e.g. ``{"large": [url1, url2], "hi_res": [...], "thumb": [...], "variant": [...]}``,
    not as a list of per-image dicts. We take the first URL of the best
    available size.
    """
    if not images_field or not isinstance(images_field, dict):
        return None
    for key in ("large", "hi_res", "thumb"):
        urls = images_field.get(key)
        if urls is not None and len(urls) > 0:
            return urls[0]
    return None


def _has_title_and_image(row: dict) -> bool:
    """Drop rows missing either a title or any usable image URL."""
    title = row.get("title")
    if not (title and title.strip()):
        return False
    return _extract_image_url(row.get("images")) is not None


def _coerce_description(desc_field) -> str:
    if isinstance(desc_field, list):
        return " ".join(str(x) for x in desc_field if x).strip()
    if isinstance(desc_field, str):
        return desc_field.strip()
    return ""


def _row_id(title: str, image_url: str) -> str:
    """Stable per-(product, image) ID: SHA-1 of title + image_url, truncated.

    Deterministic across runs so Ray Data's CheckpointConfig can resume —
    `uuid.uuid4()` regenerated fresh IDs each submission, making the
    checkpoint match zero rows on retry. Hashing on (title, image_url) keeps
    the ID unique under future multi-image fan-out (same product, different
    image → different ID).
    """
    return hashlib.sha1(f"{title}|{image_url}".encode("utf-8")).hexdigest()[:16]


def _normalize_amazon_row_to_image(row: dict) -> dict:
    """One catalog row per product, using the single best image URL (use with `.map`)."""
    title = row["title"].strip()[:512]
    image_url = _extract_image_url(row["images"])
    return {
        "id": _row_id(title, image_url),
        "product_id": row.get("parent_asin") or row.get("asin") or "",
        "title": title,
        "description": _coerce_description(row.get("description"))[:1024],
        "image_url": image_url,
        "source": "amazon-reviews-2023",
    }

def _normalize_amazon_row_to_images(row: dict, max_per_product: int = 8) -> list[dict]:
    """One catalog row per *image* — explodes a product into N rows (use with `.flat_map`).

    Takes 'large' URLs (falling back to hi_res / thumb), capped at
    ``max_per_product``. Each output row carries an ``image_idx`` so
    ``(product_id, image_idx)`` is a stable row key.
    """
    images = row.get("images") or {}
    if not isinstance(images, dict):
        return []
    urls = (
        images.get("large")
        or images.get("hi_res")
        or images.get("thumb")
        or []
    )[:max_per_product]
    if not urls:
        return []

    product_id = row.get("parent_asin") or row.get("asin") or ""
    title = row["title"].strip()[:512]
    description = _coerce_description(row.get("description"))[:1024]

    return [
        {
            "id": _row_id(title, url),
            "product_id": product_id,
            "image_idx": i,
            "title": title,
            "description": description,
            "image_url": url,
            "source": "amazon-reviews-2023",
        }
        for i, url in enumerate(urls)
    ]

if True: #not os.path.exists(cache_path):
    # load from hf
    print(f"[load hf] Loading data from huggingface hub: {hf_path}...")
    ds = ray.data.read_parquet(
            hf_path,
            file_extensions=["parquet"],
            filesystem=HfFileSystem(),
        )
    print("[load hf] count:", ds.count())
    print("[load hf] original schema:", ds.schema())

    # reduce and filter
    ds = ds.limit(n_rows)
    ds = ds.filter(_has_title_and_image)

    # naively assign 1:1 image per product, in reality would explode images across same product for 1:many
    ds = ds.map(_normalize_amazon_row_to_image)
    # ds = ds.map(_normalize_amazon_row_to_images) # untested to his is the idea
    ds = ds.random_shuffle(seed=seed)

    # cache preprocessed dataset
    ds.write_parquet(cache_path)
else:
    # cache exists already
    ds = ray.data.read_parquet(cache_path)


In [ ]:
# show example
print()
print("[load] preprocessed schema:", ds.schema())
print()

row = ds.take(1)[0]
print()
print("[load] sample row:", json.dumps(row, indent=2))
print()


resp = requests.get(row["image_url"], timeout=5,
                    headers={"User-Agent": "vlm-distillation-catalog-enrichment/1.0"})
img = Image.open(io.BytesIO(resp.content))
img.thumbnail((256, 256))
img

## Run the 7B teacher with `ray.data.llm`

[`ray.data.llm`](https://docs.ray.io/en/latest/data/working-with-llms.html#multimodal) wraps the vLLM engine as a Ray Data stage. `vLLMEngineProcessorConfig` declares the model, the per-replica engine kwargs, and the multimodal preparation stage; `build_processor` then takes a `preprocess`/`postprocess` pair and returns a callable that maps over the Dataset.

For the 7B teacher: one replica per L4 (`tensor_parallel_size=1`), `concurrency=4` replicas, `batch_size=16`. The prompt asks for a four-key JSON object; the `vlm_postprocess` step lifts `generated_text` into a `raw_output` column alongside the original row fields, then `write_parquet` materializes the result.

In [ ]:
import io, json, re
from ray.data.llm import vLLMEngineProcessorConfig, build_processor
from ray.data.checkpoint import CheckpointConfig

# Resumable batch inference: the row `id` (deterministic SHA-1 above) keys the
# checkpoint, so a re-submission picks up where the prior run left off.
ctx = ray.data.DataContext.get_current()
ctx.checkpoint_config = CheckpointConfig(
    id_column="id",
    checkpoint_path=checkpoint_path,
    delete_checkpoint_on_success=False,
)

config = vLLMEngineProcessorConfig(
    model_source="Qwen/Qwen2.5-VL-7B-Instruct",
    engine_kwargs={
        "tensor_parallel_size": 1,       # 7B fits on a single L4 at bf16
        "pipeline_parallel_size": 1,
        "max_model_len": 4096,
        "trust_remote_code": True,
        "limit_mm_per_prompt": {"image": 1},
    },
    batch_size=16,
    concurrency=4,                       # one replica per GPU on a g6.12xlarge
    prepare_multimodal_stage={"enabled": True},
    runtime_env={"worker_process_setup_hook": "src._vllm_compat.patch"},
)

prompt = """\
You are a product catalog enrichment assistant. Given a product image and \
the merchant-supplied title, output a JSON object with exactly these keys:

  category:    one short string (e.g. "Wireless Earbuds")
  attributes:  a list of 3 short attribute strings
  search_tags: a list of 5 short search keywords
  description: a single sentence (<= 30 words)

Title: {title}

Return ONLY the JSON object, no commentary.\
"""


def build_messages_url(url, title):
    """Image + text in OpenAI chat-completion format."""
    return [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": prompt.format(title=title)},
            ],
        }
    ]


def vlm_preprocess(row):
    return {
        "id": row["id"],
        "messages": build_messages_url(row["image_url"], row["title"]),
        "sampling_params": {"max_tokens": 256, "temperature": 0.0},
    }


def vlm_postprocess(row):
    return {
        "id": row["id"],                       # required for checkpoint id_column
        "product_id": row["product_id"],
        "title": row["title"],
        "image_url": row["image_url"],
        "source": row["source"],
        "raw_output": row["generated_text"],
    }


vlm_processor = build_processor(
    config,
    preprocess=vlm_preprocess,
    postprocess=vlm_postprocess,
)

ds = vlm_processor(ds)

# Show the first two enriched rows.
for output in ds.take(limit=2):
    print(json.dumps(output, indent=2))

ds.write_parquet(f"{base_dir}/teacher_7b_enriched_{n_rows}.parquet")

In [ ]:
print(ds.stats())